In [13]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import  roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import accuracy_score


Block for extracting training data and normalizing expression levels 

In [14]:
#extract data from training set
training_df = pd.read_excel("GeneData.xlsx", sheet_name="Training_Set", header=None)

#extract the gene names
gene_names = training_df.iloc[:, 0]
gene_names = gene_names.iloc[3:-1]

# extract expression data
training_data = training_df.iloc[3:, 2:].values.astype(float)
training_data = training_data[:-1] #last row is the labels - they need to be negated

#extract the labels I added to the bottom
training_labels = training_df.iloc[7132, 2:].astype(int)


# Scale the gene expression data
scaler = StandardScaler()
training_data_scaled = scaler.fit_transform(training_data)
training_data_scaled = training_data_scaled.T

Block for performing leave-out-testing to retrieve informative genes

In [3]:
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=45)
feature_importances = []

loo = LeaveOneOut()
for train_index, test_index in loo.split(training_data_scaled):    
    try:
        X, Y = training_data_scaled[train_index], training_labels.iloc[train_index]
        Y = Y.ravel()
    except IndexError as e:
        print("Error:", e)
        

    rf_classifier.fit(X, Y)
    feature_importances.append(rf_classifier.feature_importances_)
    
average_feature_importances = sum(feature_importances) / len(feature_importances)



Block for collecting weighted vote on training data (to find appropriate threshold)

Block to find best voting thresholds

In [4]:
best_auc_roc_score = 0
best_roc_thresholds = 0
best_roc_importance = 0
best_roc_numGenes = 0
best_f1_score = 0
best_f1_threshold = 0
best_f1_importance = 0
best_f1_numGenes = 0

for importance in np.arange(0.0032, 0.0064, 0.0002):
    important_feature_indicies = np.where(average_feature_importances > importance)[0]
    #reduce the data to only the informative genes
    training_informative_data = training_data_scaled[:,important_feature_indicies]
    training_informative_scaler = average_feature_importances[important_feature_indicies]
    
    
    training_informative_data = np.divide(training_informative_data.T, np.sum(training_informative_data, axis=1))
    
    training_weighted_data = np.multiply(training_informative_data.T, training_informative_scaler)
    
    training_weighted_votes = np.sum(training_weighted_data, axis=1)
    
    max_vote = training_weighted_votes[np.argmax(training_weighted_votes)]
    min_vote = training_weighted_votes[np.argmin(training_weighted_votes)]

    step = (max_vote - min_vote )/20
    
    numGenes = important_feature_indicies.size
    
    for threshold in np.arange(min_vote, max_vote, step):
        results = (training_weighted_votes > threshold).astype(int)
        auc_roc = roc_auc_score(training_labels, results)
        f1 = f1_score(training_labels, results)
        
        print("\nThreshold: ", threshold)
        print("Area under ROC: ", auc_roc)
        print("F1 score: ", f1)
        
        positive_accuracy = len(np.where(results == 1)[0]) / (len(np.where(results == 1)[0]) + len(np.where(training_labels == 1)[0]))
        negative_accuracy = len(np.where(results == 0)[0]) / (len(np.where(results == 0)[0]) + len(np.where(training_labels == 0)[0]))
        
        if f1 > best_f1_score:
            if (positive_accuracy > 0.4) & (negative_accuracy > 0.4):
                best_f1_score = f1
                best_f1_threshold = threshold
                best_f1_importance = importance
                best_f1_numGenes = numGenes
            
        
        if auc_roc > best_auc_roc_score:
            if (positive_accuracy > 0.4) & (negative_accuracy > 0.4):
                best_auc_roc_score = auc_roc
                best_roc_threshold = threshold
                best_roc_importance = importance
                best_roc_numGenes = numGenes
        
print("best roc threshold ", best_roc_threshold)
print("best roc #genes ", best_roc_numGenes)

print("best f1 threshold ", best_f1_threshold)
print("best f1 #genes ", best_f1_numGenes)


Threshold:  0.0035416800549873204
Area under ROC:  0.55
F1 score:  0.8163265306122449

Threshold:  0.003591198492113094
Area under ROC:  0.55
F1 score:  0.7826086956521738

Threshold:  0.0036407169292388677
Area under ROC:  0.525
F1 score:  0.7555555555555556

Threshold:  0.0036902353663646414
Area under ROC:  0.475
F1 score:  0.6976744186046512

Threshold:  0.003739753803490415
Area under ROC:  0.525
F1 score:  0.7142857142857143

Threshold:  0.0037892722406161886
Area under ROC:  0.55
F1 score:  0.7

Threshold:  0.0038387906777419623
Area under ROC:  0.5
F1 score:  0.631578947368421

Threshold:  0.003888309114867736
Area under ROC:  0.5
F1 score:  0.5714285714285715

Threshold:  0.00393782755199351
Area under ROC:  0.44999999999999996
F1 score:  0.4848484848484849

Threshold:  0.003987345989119283
Area under ROC:  0.49999999999999994
F1 score:  0.41379310344827586

Threshold:  0.004036864426245057
Area under ROC:  0.49999999999999994
F1 score:  0.41379310344827586

Threshold:  0.004

Block to print results

In [5]:
important_feature_indicies = np.where(average_feature_importances > best_roc_importance)[0]
#reduce the data to only the informative genes
training_informative_data = training_data_scaled[:,important_feature_indicies]
training_informative_scaler = average_feature_importances[important_feature_indicies]

training_informative_data = np.divide(training_informative_data.T, np.sum(training_informative_data, axis=1))

training_weighted_data = np.multiply(training_informative_data.T, training_informative_scaler)

training_weighted_votes = np.sum(training_weighted_data, axis=1)

numGenes = important_feature_indicies.size

results = (training_weighted_votes > best_roc_threshold).astype(int)

auc_roc = roc_auc_score(training_labels, results)
precision = precision_score(training_labels, results)
recall = recall_score(training_labels, results)
f1 = f1_score(training_labels, results)
conf_matrix = confusion_matrix(training_labels, results)
accuracy = accuracy_score(training_labels, results)

print("MAXIMIZING AREA UNDER ROC")
print(f"Threshold: {best_roc_threshold}")
print(f"Number of genes: {numGenes}")
print(f"Area under ROC: {auc_roc}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")
print(f"Accuracy Score: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)

#maximize f1

important_feature_indicies = np.where(average_feature_importances > best_f1_importance)[0]
#reduce the data to only the informative genes
training_informative_data = training_data_scaled[:,important_feature_indicies]
training_informative_scaler = average_feature_importances[important_feature_indicies]

training_informative_data = np.divide(training_informative_data.T, np.sum(training_informative_data, axis=1))

training_weighted_data = np.multiply(training_informative_data.T, training_informative_scaler)

training_weighted_votes = np.sum(training_weighted_data, axis=1)

numGenes = important_feature_indicies.size

results = (training_weighted_votes > best_f1_threshold).astype(int)
auc_roc = roc_auc_score(training_labels, results)
precision = precision_score(training_labels, results)
recall = recall_score(training_labels, results)
f1 = f1_score(training_labels, results)
conf_matrix = confusion_matrix(training_labels, results)
accuracy = accuracy_score(training_labels, results)

print("\nMAXIMIZING F1 SCORE ")
print(f"Threshold: {best_f1_threshold}")
print(f"Number of genes: {numGenes}")
print(f"Area under ROC: {auc_roc}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")
print(f"Accuracy Score: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)

MAXIMIZING AREA UNDER ROC
Threshold: 0.0037892722406161886
Number of genes: 50
Area under ROC: 0.55
Precision: 0.7
Recall: 0.7
F1 Score: 0.7
Accuracy Score: 0.6
Confusion Matrix:
[[ 4  6]
 [ 6 14]]

MAXIMIZING F1 SCORE 
Threshold: 0.003739753803490415
Number of genes: 50
Area under ROC: 0.525
Precision: 0.6818181818181818
Recall: 0.75
F1 Score: 0.7142857142857143
Accuracy Score: 0.6
Confusion Matrix:
[[ 3  7]
 [ 5 15]]


Extract testing data

In [7]:
#extract data from training set
testing_df = pd.read_excel("GeneData.xlsx", sheet_name="Testing_Set", header=None)

# extract expression data
testing_data = testing_df.iloc[3:, 2:].values.astype(float)
testing_data = testing_data[:-1] #last row is the labels - they need to be negated

#extract the labels I added to the bottom
testing_labels = testing_df.iloc[7132, 2:].astype(int)


# Scale the gene expression data
scaler = StandardScaler()
testing_data_scaled = scaler.fit_transform(testing_data)
testing_data_scaled = testing_data_scaled.T

In [8]:
#reduce the data to only the informative genes
important_feature_indicies = np.where(average_feature_importances > best_roc_importance)[0]
testing_informative_data = testing_data_scaled[:,important_feature_indicies]
testing_informative_scaler = average_feature_importances[important_feature_indicies]


testing_informative_data = np.divide(testing_informative_data.T, np.sum(testing_informative_data, axis=1))

testing_weighted_data = np.multiply(testing_informative_data.T, testing_informative_scaler)

testing_weighted_votes = np.sum(testing_weighted_data, axis=1)

In [9]:
results = (testing_weighted_votes > best_roc_threshold).astype(int)

precision = precision_score(testing_labels, results)
recall = recall_score(testing_labels, results)
f1 = f1_score(testing_labels, results)
conf_matrix = confusion_matrix(testing_labels, results)
accuracy = accuracy_score(testing_labels, results)
print("threshold", best_roc_threshold)
print(f"Area under ROC: {auc_roc}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")
print(f"Accuracy Score: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)

threshold 0.0037892722406161886
Area under ROC: 0.525
Precision: 0.7058823529411765
Recall: 0.631578947368421
F1 Score: 0.6666666666666667
Accuracy Score: 0.6
Confusion Matrix:
[[ 6  5]
 [ 7 12]]


Performing the same experiment with Random Forest Model

In [44]:
best_auc_roc_score = 0
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=50)

for importance in np.arange(0.0032, 0.0064, 0.0008):
    important_feature_indicies = np.where(average_feature_importances > importance)[0]
    training_informative_data = training_data_scaled[:,important_feature_indicies]
    rf_classifier.fit(training_informative_data, training_labels)
    
    testing_informative_data = testing_data_scaled[:,important_feature_indicies]
    predictions = rf_classifier.predict(testing_weighted_data)
    auc_roc = roc_auc_score(testing_labels, predictions)
    
    positive_accuracy = len(np.where(predictions == 1)[0]) / (len(np.where(predictions == 1)[0]) + len(np.where(training_labels == 1)[0]))
    negative_accuracy = len(np.where(predictions == 0)[0]) / (len(np.where(results == 0)[0]) + len(np.where(training_labels == 0)[0]))

    if auc_roc > best_auc_roc_score:
        if (positive_accuracy > 0.4) & (negative_accuracy > 0.4):
            best_auc_roc_score = auc_roc
            best_roc_importance = importance




In [46]:
important_feature_indicies = np.where(average_feature_importances > importance)[0]
training_informative_data = training_data_scaled[:,important_feature_indicies]
rf_classifier.fit(training_informative_data, training_labels)

testing_informative_data = testing_data_scaled[:,important_feature_indicies]
predictions = rf_classifier.predict(testing_weighted_data)

precision = precision_score(testing_labels, predictions)
numGenes = important_feature_indicies.size
recall = recall_score(testing_labels, predictions)
f1 = f1_score(testing_labels, predictions)
conf_matrix = confusion_matrix(testing_labels, predictions)
accuracy = accuracy_score(testing_labels, predictions)

print(f"Number of genes: {numGenes}")
print(f"Area under ROC: {auc_roc}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")
print(f"Accuracy Score: {accuracy}")
print("Confusion Matrix:")
print(conf_matrix)

Area under ROC: 0.525
Precision: 0.6
Recall: 0.7894736842105263
F1 Score: 0.6818181818181819
Accuracy Score: 0.5333333333333333
Confusion Matrix:
[[ 1 10]
 [ 4 15]]
